# SawitGuard-GNN — Lapisan 1 · Tahap 1: deteksi tajuk dengan **YOLOv12**

Notebook kerja untuk tahap deteksi Lapisan 1: melatih YOLOv12 pada 2.303 ubin UAV nadir RGB
(dataset B, 1024², kelas `Healthy` / `Unhealthy`), dengan ablasi augmentasi dan slot untuk
ubin dari sumber luar. Seluruh logikanya ada di `y12.py`; notebook memanggilnya, tidak
menyalinnya, jadi keduanya tidak mungkin berbeda isi.

### Metrik utamanya adalah PUSAT TAJUK, bukan mAP — dan itu keputusan berbasis bukti

Bagian 2 mengukur mutu label dataset ini. Ringkas hasilnya: **kotak kebenaran-dasar ds_B
adalah cap berukuran tetap, bukan kotak yang digambar per tajuk** — hanya 23–30 ukuran kotak
berbeda untuk 1.379–1.849 tajuk, rasio aspek terkunci di 0,99, dan 33–42% kotak lebih besar
daripada jarak tanam. Akibatnya mAP50-95 punya langit-langit yang **tidak dapat dinaikkan oleh
model, epoch, maupun augmentasi apa pun**, karena detektor yang menggambar batas tajuk
sebenarnya justru dihukum oleh label cap tersebut.

Yang tetap dapat dipercaya dari label ini adalah **pusatnya**, dan pusat itulah yang
dikonsumsi Lapisan 2. Karena itu metrik utama notebook ini adalah presisi/recall/F1 pusat
tajuk pada radius pencocokan, ditambah RMSE pusat — dihitung pada **pohon unik** setelah
deteksi dari ubin bertindih digabungkan. mAP tetap dilaporkan, sebagai angka pembanding
dengan langit-langitnya dinyatakan.

### Disiplin yang dikunci sebelum satu angka pun muncul

| | |
|---|---|
| Evaluasi | **leave-one-ortho-out**, 3 ortomosaik. Split acak bocor ~100% (diukur di bagian 2) dan **tidak disediakan** oleh `y12.py` |
| Metrik utama | **F1 pusat tajuk + RMSE pusat**, pada pohon unik |
| Metrik sekunder | mAP50 / mAP50-95 / AP per kelas, dengan langit-langit label dinyatakan |
| Aturan putus | selisih di dalam **satu simpangan baku (ddof=1)** ⇒ **TIDAK KONKLUSIF** — sama persis dengan `run_experiment.py::paired` di Lapisan 2 |
| Urutan | garis dasar dulu, ablasi kemudian. Augmentasi harus **membuktikan diri** |

### Yang notebook ini TIDAK bisa klaim, apa pun hasilnya

1. **Label bukan BSR.** `Unhealthy` adalah kesehatan tajuk generik tanpa verifikasi lapangan
   (`../layer1_data_audit/AUDIT_REPORT.md`). Detektor yang bagus di sini tetap **bukan**
   detektor Ganoderma.
2. **n = 3 ortomosaik, satu kebun.** Sebagian besar selisih akan jatuh TIDAK KONKLUSIF. Itu
   jawaban yang benar pada ukuran sampel ini, bukan kegagalan.
3. **Positif sangat langka:** ≈66 pohon `Unhealthy` unik di seluruh dataset. Angka apa pun
   untuk kelas itu berderau lebar dan wajib dilaporkan terpisah.

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np

sys.path.insert(0, os.getcwd())
import y12
from y12 import ARMS, NOT_USED

import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("akar data  ", y12.ROOT)

## 0. Setelan

Ubah **di sini saja**. Semua sel berikutnya membaca konstanta ini, jadi tidak ada setelan yang
tersembunyi di tengah notebook.

`SEEDS` bukan hiasan: dengan hanya 3 ortomosaik, satu seed memberi 3 pasangan dan hampir semua
selisih otomatis TIDAK KONKLUSIF. Dua atau tiga seed memberi 6–9 pasangan dan membuat hitungan
tanda punya arti. **Seed menambah pasangan, bukan situs** — ia mempersempit derau optimisasi,
bukan ketidakpastian antar-kebun, dan klaim generalisasi tetap dibatasi n = 3.

In [ ]:
MODEL        = "yolo12n.pt"      # yolo12s.pt bila VRAM dan waktu mengizinkan
IMGSZ        = 640               # ubin asli 1024; 640 = kompromi waktu/VRAM 8 GB
EPOCHS_ABL   = 30                # untuk ablasi augmentasi (semua lengan WAJIB sama)
EPOCHS_FINAL = 60                # untuk model akhir saja
SEEDS        = (42,)             # -> (42, 7) untuk 6 pasangan; biaya berlipat
ARMS_RUN     = ["base", "nadir", "nadir_geom", "nadir_nohsv", "nadir_strong"]
EXTRA_MODE   = "holdout"         # 'ignore' | 'holdout' | 'train'  (lihat bagian 3)
CACHE        = "ram"             # False bila RAM sempit di IMGSZ 1024
WORKERS      = 0                 # WAJIB 0 di notebook Windows (spawn dataloader menggantung)

CONF         = 0.25              # ambang keyakinan untuk evaluasi pusat tajuk
RADIUS_FRAC  = 0.5               # radius cocok = 0,5 x jarak tanam

# Perkiraan kasar; kalibrasikan setelah lengan pertama selesai.
MIN_PER_EPOCH_FOLD = 0.45        # menit, yolo12n @640 di RTX 5060 Laptop
est = MIN_PER_EPOCH_FOLD * EPOCHS_ABL * 3 * len(SEEDS) * len(ARMS_RUN)
print("lengan ablasi : %d x 3 lipatan x %d seed x %d epoch"
      % (len(ARMS_RUN), len(SEEDS), EPOCHS_ABL))
print("perkiraan     : %.1f jam ablasi + %.1f jam model akhir"
      % (est / 60, MIN_PER_EPOCH_FOLD * EPOCHS_FINAL * 3 / 60))
print("\nHasil disimpan per-lari ke yolo12_results/. Notebook boleh dihentikan dan")
print("dilanjutkan: train_arm() melewati kombinasi (lipatan, seed) yang sudah ada.")

## 1. Data dan harness

`y12.build()` menautkan (hardlink, bukan salin) ubin dataset B ke `yolo12/`, menulis label YOLO,
lalu membuat satu berkas yaml per lipatan **leave-one-ortho-out**. Region diambil dari prefiks
nama berkas Roboflow (`44000_16000_...`) — itulah blok spasialnya.

In [ ]:
folds = y12.build(extra_mode=EXTRA_MODE)

In [ ]:
bal = y12.class_balance(folds)
print("%-8s %7s %10s %11s %8s" % ("lipatan", "citra", "Healthy", "Unhealthy", "% Unh"))
for r in bal:
    print("%-8s %7d %10d %11d %7.2f%%"
          % (r["fold"], r["images"], r["healthy"], r["unhealthy"], r["pct_unhealthy"]))
print("\nKotak-kotak ini adalah tampilan berulang dari hanya ~66 pohon Unhealthy unik")
print("(../data_clean/DATASET_CARD.md) - itulah ukuran sampel yang sebenarnya.")

## 2. Audit mutu label — mengapa mAP bukan metrik utama

Tiga pengukuran. Ketiganya dijalankan dari `y12.py`, jadi angkanya dapat dikutip dan diulang.

**(a) Kotak adalah cap, bukan gambar.** Bila hanya ada belasan ukuran kotak untuk ribuan tajuk
dan rasio aspeknya terkunci, kotak itu ditempelkan pada pusat, bukan digambar mengikuti tajuk.
Konsekuensinya langsung: mAP50-95 mengukur seberapa baik model **meniru palet cap**, bukan
seberapa baik ia menemukan tajuk.

**(b) Redundansi piksel.** Ubin diambil pada offset acak dan saling bertindih. "Jumlah citra
latih" karena itu jauh melebihi jumlah informasi yang tersedia.

**(c) Kebocoran split bawaan.** Karena satu pohon muncul di puluhan ubin, split acak bawaan
Roboflow menaruh pohon yang sama di train dan test. Inilah sebabnya papan skor Roboflow tampak
nyaris sempurna — angka itu mengukur hafalan, bukan generalisasi, dan **tidak sebanding** dengan
angka block-CV mana pun di notebook ini.

In [ ]:
print("(a) UKURAN KOTAK")
print("  %-14s %7s %10s %10s %8s %9s %11s %11s"
      % ("ortho", "pohon", "jarak tnm", "kotak med", "CV", "n ukuran", "5 teratas", "kotak>jarak"))
for r in y12.label_audit():
    print("  %-14s %7d %10.1f %10.1f %8.3f %9d %10.1f%% %10.1f%%"
          % (r["ortho"], r["trees"], r["spacing_px"], r["box_med_px"], r["box_cv"],
             r["n_unique_sizes"], 100 * r["top5_share"], 100 * r["frac_box_gt_spacing"]))
    print("      ukuran terbanyak: %s" % ", ".join(r["top_sizes"]))
print("\n  Rasio aspek terkunci + belasan ukuran unik = cap berukuran tetap.")
print("  -> mAP50-95 punya langit-langit yang tidak bergantung pada model.")
print("  -> 33-42% kotak lebih besar dari jarak tanam: kotak tetangga bertindih")
print("     secara bawaan, dan NMS memotong recall. Ini juga bukan cacat model.")

In [ ]:
print("(b) REDUNDANSI PIKSEL")
tot_u = tot_s = 0.0
for r in y12.redundancy_audit():
    tot_u += r["unique_mpx"]; tot_s += r["summed_mpx"]
    print("  %-14s ubin=%4d  unik=%5.1f Mpx  dijumlah=%6.1f Mpx  redundansi=%.1fx"
          % (r["ortho"], r["tiles"], r["unique_mpx"], r["summed_mpx"], r["redundancy"]))
print("  %-14s %10s unik=%5.1f Mpx  dijumlah=%6.1f Mpx  redundansi=%.1fx"
      % ("TOTAL", "", tot_u, tot_s, tot_s / tot_u))
print("\n  Melatih pada ~1.500 'citra' per lipatan = melatih pada ~%.0f Mpx tanah unik." % tot_u)
print("  Epoch tambahan tidak menambah informasi yang tidak ada di %.0f Mpx itu." % tot_u)

In [ ]:
print("(c) KEBOCORAN SPLIT BAWAAN ROBOFLOW")
print("  %-14s %8s %16s %16s" % ("ortho", "pohon", "valid ada di train", "test ada di train"))
la = y12.leak_audit()
for r in la:
    print("  %-14s %8d %15.1f%% %15.1f%%"
          % (r["ortho"], r["trees"], 100 * r["valid_in_train"], 100 * r["test_in_train"]))
print("  %-14s %8s %15.1f%% %15.1f%%"
      % ("rata-rata", "", 100 * np.mean([r["valid_in_train"] for r in la]),
         100 * np.mean([r["test_in_train"] for r in la])))
print("\n  Skor tinggi pada split bawaan = model dinilai pada pohon yang melatihnya.")
print("  Itulah sebabnya notebook ini membuang split bawaan dan memakai block-CV")
print("  per-ortomosaik. Dua angka itu TIDAK boleh dibandingkan satu sama lain.")

## 3. Ubin tambahan dari sumber luar (opsional)

Taruh ubin di `extra/images/` dan label YOLO-nya di `extra/labels/` (`<stem>.txt`, kelas
`0`=Healthy `1`=Unhealthy). Citra tanpa label **dilewati** dan dilaporkan — citra tak berlabel
bukan data latih. Rinciannya di `extra/README.md`.

**Sebelum menambahkan tangkapan layar Google Maps/Earth, tiga hal yang harus diputuskan sadar:**

1. **Lisensi.** Ketentuan layanan Google melarang pengambilan ubin massal dan pembuatan dataset
   turunan dari citranya. Ubin seperti itu tidak dapat didistribusikan bersama paper dan
   sebaiknya tidak masuk angka yang dilaporkan. Citra udara berlisensi terbuka adalah masukan
   yang bersih untuk slot yang sama ini.
2. **Domain berbeda.** Citra satelit konsumen ~0,15–0,5 m/px melawan UAV 0,087 m/px. Sel di
   bawah mengukur diameter tajuk dalam piksel supaya ketidakcocokan skala terlihat sebagai
   angka, bukan dugaan.
3. **Label.** Tangkapan layar datang tanpa anotasi. Melabeli sehat/sakit dengan mata dari citra
   satelit menambah data sekaligus menambah **sumber derau label baru yang tidak terukur** —
   pada dataset yang, seperti baru saja diukur, sudah punya masalah mutu label.

| mode | perlakuan | pertanyaan yang dijawab |
|---|---|---|
| `ignore` | tidak dipakai | — |
| `holdout` | lipatan tambahan `foldX`, **hanya diuji** | apakah model 3-ortho bertahan di domain lain? |
| `train` | ditambah ke train **setiap** lipatan, tak pernah ke val | apakah data tambahan menaikkan performa pada 3 ortomosaik nyata? |

Tidak ada mode yang mengaduk ubin luar secara acak ke train+val — itu akan mengulang persis
kebocoran yang baru saja diukur di bagian 2c.

In [ ]:
sc = y12.scale_check(sample=150)
print("%-14s %6s %8s %12s %10s %10s" % ("region", "ubin", "kotak", "diag med px", "p10", "p90"))
for r in sc:
    print("%-14s %6d %8d %12.1f %10.1f %10.1f"
          % (r["region"], r["tiles"], r["boxes"], r["diag_med_px"], r["diag_p10"], r["diag_p90"]))
ref = np.median([r["diag_med_px"] for r in sc if r["region"] != y12.EXTRA_REGION])
ex = [r for r in sc if r["region"] == y12.EXTRA_REGION]
print("\nacuan ds_B: diameter tajuk median %.0f px pada ubin 1024." % ref)
if ex:
    rel = ex[0]["diag_med_px"] / ref
    print("EXTRA     : %.0f px = %.2fx acuan." % (ex[0]["diag_med_px"], rel))
    print("  -> %s" % ("skala sepadan, boleh lanjut." if 0.8 <= rel <= 1.25 else
          "TIDAK sepadan. Resample dulu, atau pakai mode 'holdout' saja dan laporkan "
          "sebagai uji lintas-domain - jangan campurkan ke train."))
else:
    print("EXTRA     : belum ada ubin berlabel. Bagian ini tidak berpengaruh.")

## 4. Melihat augmentasi, bukan mengasumsikannya

Sel ini menarik citra latih **setelah** augmentasi lengan benar-benar diterapkan. Yang dicari:
apakah mosaic memotong tajuk di jahitan, dan apakah jitter HSV menghapus perbedaan warna yang
justru menjadi sinyal kelas `Unhealthy` (fitur teratas Tahap 2 adalah G−R, exg_std, G−B —
seluruhnya greenness).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SHOW = ["base", "nadir", "nadir_strong"]
fig, axes = plt.subplots(len(SHOW), 4, figsize=(15, 3.9 * len(SHOW)))
for r, arm in enumerate(SHOW):
    for c, (im, box, cls) in enumerate(y12.preview_aug(arm, n=4, imgsz=IMGSZ)):
        ax = axes[r, c]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
        H, W = im.shape[:2]
        for (cx, cy, bw, bh), k in zip(box, cls):
            ax.add_patch(mpatches.Rectangle(((cx - bw / 2) * W, (cy - bh / 2) * H),
                                            bw * W, bh * H, fill=False, lw=0.8,
                                            edgecolor="yellow" if k == 0 else "red"))
        if c == 0:
            ax.set_ylabel(arm, fontsize=11)
        ax.set_title("%d kotak" % len(cls), fontsize=8)
plt.suptitle("Citra latih setelah augmentasi  (kuning=Healthy, merah=Unhealthy)", y=0.995)
plt.tight_layout(); plt.show()

print("Augmentasi yang SENGAJA tidak dipakai, dan alasannya:")
for k, v in NOT_USED.items():
    print("  %-14s %s" % (k, v))

## 5. Melatih — garis dasar dulu, lalu ablasi

Setiap lengan dijalankan pada lipatan dan seed yang identik. `train_arm()` menyimpan hasil
setelah setiap lari, jadi notebook boleh dihentikan kapan saja dan dilanjutkan tanpa mengulang
apa yang sudah selesai.

Isi tiap lengan dan **alasan fisiknya** tercetak di bawah. Lengan tanpa alasan fisik hanyalah
pencarian hiperparameter yang menyamar; pada dataset dengan ~71 Mpx tanah unik, itu cara
tercepat menipu diri sendiri.

In [ ]:
for a in ARMS_RUN:
    print("=" * 78)
    print("%-14s %s" % (a, {k: v for k, v in ARMS[a].items() if not k.startswith("_")}
                        or "(default)"))
    print("               %s" % ARMS[a]["_why"])

In [ ]:
t0 = time.time()
for a in ARMS_RUN:
    print("\n" + "#" * 78 + "\n# lengan: %s\n" % a + "#" * 78, flush=True)
    y12.train_arm(a, model=MODEL, folds=folds, seeds=SEEDS, epochs=EPOCHS_ABL,
                  imgsz=IMGSZ, cache=CACHE, workers=WORKERS)
print("\nablasi selesai dalam %.1f menit" % ((time.time() - t0) / 60))

## 6. Putusan ablasi (metrik sekunder)

Selisih berpasangan per (lipatan, seed) terhadap `base`, dengan aturan yang sama dengan
Lapisan 2. `foldX` dikeluarkan dari rata-rata ini.

Angka di sini adalah mAP, jadi bacalah dengan langit-langit bagian 2 dalam kepala: perbedaan
antar-lengan pada mAP50-95 sebagian mengukur seberapa baik tiap lengan meniru palet cap.
Putusan akhir tetap dikonfirmasi pada metrik pusat tajuk di bagian 8.

`TIDAK KONKLUSIF` berarti data ini tidak dapat membedakan lengan itu dari garis dasar — **bukan**
bahwa lengan itu diam-diam lebih baik.

In [ ]:
rows_map = y12.report(key="map", base=None)
print()
rows_map50 = y12.report(key="map50", base=None)

In [ ]:
runs = y12.load_results()
print("%-24s %s" % ("lengan", "AP50 Unhealthy per (lipatan|seed)"))
for tag in sorted(runs):
    v = {k: r["ap50"].get("Unhealthy", float("nan"))
         for k, r in runs[tag]["runs"].items() if not k.startswith("foldX")}
    arr = np.array(list(v.values()))
    print("%-24s mean %.3f +/- %.3f   %s"
          % (tag, arr.mean(), arr.std(ddof=1) if len(arr) > 1 else float("nan"),
             " ".join("%s=%.3f" % kv for kv in sorted(v.items()))))
print("\nAngka ini bersandar pada ~66 pohon Unhealthy unik di seluruh dataset.")
print("Selisih antar-lengan di sini hampir pasti derau; laporkan apa adanya.")

In [ ]:
gaps = [g for g in (y12.domain_gap(t) for t in sorted(runs)) if g]
if gaps:
    print("%-24s %12s %10s %10s" % ("lengan", "3-ortho mAP", "EXTRA", "selisih"))
    for g in gaps:
        print("%-24s %12.3f %10.3f %+10.3f" % (g["tag"], g["in_domain"], g["extra"], g["gap"]))
    print("\nSelisih negatif besar = model tidak berpindah domain. Itu hasil yang sah dan")
    print("wajib dilaporkan; ia TIDAK membatalkan angka 3-ortho, ia membatasi klaimnya.")
else:
    print("Tidak ada blok EXTRA. Lewati.")

## 7. Model akhir

Lengan pemenang dipilih dengan aturan eksplisit: **ambil lengan yang putusannya `POSITIF`;
bila tidak ada, tetap `base`.** Aturan ini ditulis sebelum hasilnya dilihat. Memilih lengan
ber-mean tertinggi di antara yang TIDAK KONKLUSIF adalah memungut derau.

Lari akhir memakai `tag_suffix="final"` supaya hasilnya tidak menimpa hasil ablasi lengan yang
sama — jumlah epoch-nya berbeda, jadi ia bukan anggota perbandingan bagian 6.

In [ ]:
pos = [r for r in rows_map if r.get("verdict") == "POSITIF"]
best_tag = max(pos, key=lambda r: r["delta"])["tag"] if pos else \
           next(t for t in runs if t.endswith("_base"))
best_arm = runs[best_tag]["arm"]
FINAL_TAG = "%s_%s_final" % (os.path.splitext(MODEL)[0], best_arm)
print("lengan terpilih: %s  (%s)" % (best_arm,
      "menang secara konklusif" if pos else "tidak ada lengan yang lolos pita derau -> garis dasar"))

y12.train_arm(best_arm, model=MODEL, folds=folds, seeds=SEEDS, epochs=EPOCHS_FINAL,
              imgsz=IMGSZ, cache=CACHE, workers=WORKERS, resume_ok=False, tag_suffix="final")

In [ ]:
import glob, shutil
os.makedirs(os.path.join(y12.BASE, "yolo12_final"), exist_ok=True)
for p in sorted(glob.glob(os.path.join(y12.RUNS, FINAL_TAG + "_fold*_s*",
                                       "weights", "best.pt"))):
    dst = os.path.join(y12.BASE, "yolo12_final",
                       os.path.basename(os.path.dirname(os.path.dirname(p))) + ".pt")
    shutil.copy(p, dst)
    print("tersimpan", os.path.basename(dst))
print("\nSatu bobot per lipatan, sengaja. Tidak ada bobot 'final tunggal' yang sah:")
print("tiap bobot hanya boleh dinilai pada ortomosaik yang TIDAK ikut melatihnya.")

## 8. Metrik utama — pusat tajuk pada pohon unik

Untuk tiap lipatan: jalankan bobotnya pada seluruh ubin ortomosaik yang ditahan, petakan pusat
kotak ke koordinat global, **gabungkan deteksi duplikat** (ubin bertindih ~34×), lalu cocokkan
satu-ke-satu dengan pohon unik dari `layer1_crowns.csv` pada radius `RADIUS_FRAC` × jarak tanam.

Dua hal yang membuat metrik ini lebih jujur daripada mAP di sini:

1. **Tidak dibatasi geometri cap.** Ia menilai posisi, dan posisi adalah bagian label yang
   memang dapat dipercaya.
2. **Dievaluasi pada pohon unik.** mAP per-ubin merata-ratakan ~34 tampilan berkorelasi dari
   pohon yang sama, sehingga simpangan bakunya terlalu sempit. Di sini tiap pohon dihitung
   sekali.

Sapuan radius di bawahnya adalah pemeriksaan kejujurannya sendiri: kurva yang **datar** berarti
pusatnya memang tepat; kurva yang **menanjak tajam** berarti "benar" hanya karena ambangnya
dilonggarkan.

In [ ]:
cen = []
for f in folds:
    if f == "foldX":
        continue
    wp = os.path.join(y12.BASE, "yolo12_final", "%s_%s_s%d.pt" % (FINAL_TAG, f, SEEDS[0]))
    cen.append(y12.centre_eval(wp, f, radius_frac=RADIUS_FRAC, conf=CONF, imgsz=IMGSZ))

f1 = np.array([c["f1"] for c in cen]); pr = np.array([c["precision"] for c in cen])
rc = np.array([c["recall"] for c in cen]); rm = np.array([c["rmse_frac_spacing"] for c in cen])
print("\n%-12s %8s %8s %8s %12s" % ("", "P", "R", "F1", "RMSE/jarak"))
for c in cen:
    print("%-12s %8.3f %8.3f %8.3f %12.3f"
          % (c["ortho"], c["precision"], c["recall"], c["f1"], c["rmse_frac_spacing"]))
print("%-12s %8.3f %8.3f %8.3f %12.3f   <- mean" % ("", pr.mean(), rc.mean(), f1.mean(), rm.mean()))
print("%-12s %8.3f %8.3f %8.3f %12.3f   <- std (n=3 ortomosaik)"
      % ("", pr.std(ddof=1), rc.std(ddof=1), f1.std(ddof=1), rm.std(ddof=1)))
print("\nUnhealthy ditemukan: %s dari %s"
      % ([c["unhealthy_found"] for c in cen], [c["unhealthy_gt"] for c in cen]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for f in folds:
    if f == "foldX":
        continue
    wp = os.path.join(y12.BASE, "yolo12_final", "%s_%s_s%d.pt" % (FINAL_TAG, f, SEEDS[0]))
    sw = y12.radius_sweep(wp, f, conf=CONF, imgsz=IMGSZ)
    ax[0].plot([s["frac"] for s in sw], [s["f1"] for s in sw], "o-", label=f)
    ax[1].plot([s["frac"] for s in sw], [s["precision"] for s in sw], "o-", label=f + " P")
    ax[1].plot([s["frac"] for s in sw], [s["recall"] for s in sw], "s--", label=f + " R")
for a, t in zip(ax, ["F1 vs radius pencocokan", "Presisi & recall vs radius"]):
    a.set_xlabel("radius / jarak tanam"); a.set_title(t); a.grid(alpha=.3); a.legend(fontsize=8)
ax[0].set_ylabel("F1")
plt.tight_layout(); plt.show()
print("Kurva datar mulai ~0,35 = pusat benar-benar tepat, bukan hasil ambang longgar.")

## 9. Prediksi kualitatif

Gambar, bukan angka. Berguna untuk melihat mode kegagalan (tajuk bertumpuk, tepi ubin, bayangan)
yang tidak muncul di angka mana pun.

In [ ]:
from ultralytics import YOLO
fold_show = folds[0]
wp = os.path.join(y12.BASE, "yolo12_final", "%s_%s_s%d.pt" % (FINAL_TAG, fold_show, SEEDS[0]))
m = YOLO(wp)
val_imgs = [l.strip() for l in open(os.path.join(y12.ROOT, "%s_val.txt" % fold_show))
            if l.strip()][:6]
res = m.predict(val_imgs, imgsz=IMGSZ, conf=CONF, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax_, src, r in zip(axes.ravel(), val_imgs, res):
    ax_.imshow(r.plot()[:, :, ::-1]); ax_.set_xticks([]); ax_.set_yticks([])
    ax_.set_title(os.path.basename(src)[:34], fontsize=8)
plt.suptitle("Prediksi pada ortomosaik yang DITAHAN (%s) — bobot %s" % (fold_show, best_arm))
plt.tight_layout(); plt.show()

## 10. Ringkasan dan klaim maksimum

Sel terakhir merakit kalimat yang **boleh** ditulis di paper dari angka yang benar-benar ada di
`yolo12_results/`. Bila sebuah angka tidak muncul di sini, ia tidak boleh muncul di naskah
(`../paper/METHODOLOGY_PLAN.md` §7).

In [ ]:
fin = y12.load_results(only=[FINAL_TAG])[FINAL_TAG]
keys = [k for k in fin["runs"] if not k.startswith("foldX")]
st = fin["setting"]
mm = np.array([fin["runs"][k]["map"] for k in keys])
mm50 = np.array([fin["runs"][k]["map50"] for k in keys])
unh = np.array([fin["runs"][k]["ap50"].get("Unhealthy", np.nan) for k in keys])
abl = next(r for r in rows_map if r["tag"] == best_tag)
sd = lambda a: float(a.std(ddof=1)) if len(a) > 1 else float("nan")

verd = ("lengan '%s' menang konklusif (+%.4f mAP50-95 pada ablasi %d epoch)"
        % (best_arm, abl.get("delta", 0.0), runs[best_tag]["setting"]["epochs"]) if pos else
        "tidak satu lengan pun keluar dari pita derau; garis dasar dipertahankan dan "
        "augmentasi TIDAK boleh disebut sebagai peningkatan")

claim = [
 "Deteksi tajuk sawit dari citra UAV RGB nadir dengan YOLOv12 (%s, %d epoch, imgsz %d),"
 % (MODEL, st["epochs"], st["imgsz"]),
 "dievaluasi dengan validasi silang blok leave-one-ortho-out pada 3 ortomosaik satu kebun.",
 "",
 "METRIK UTAMA - pusat tajuk pada pohon unik (radius cocok %.2f x jarak tanam):" % RADIUS_FRAC,
 "  presisi %.3f +/- %.3f | recall %.3f +/- %.3f | F1 %.3f +/- %.3f"
 % (pr.mean(), sd(pr), rc.mean(), sd(rc), f1.mean(), sd(f1)),
 "  RMSE pusat %.3f +/- %.3f x jarak tanam" % (rm.mean(), sd(rm)),
 "",
 "Metrik sekunder - mAP50 %.3f +/- %.3f, mAP50-95 %.3f +/- %.3f, AP50 Unhealthy %.3f."
 % (mm50.mean(), sd(mm50), mm.mean(), sd(mm), np.nanmean(unh)),
 "mAP50-95 dilaporkan DENGAN langit-langitnya: kotak kebenaran-dasar ds_B hanya",
 "punya 23-30 ukuran berbeda untuk 1.379-1.849 tajuk dengan rasio aspek terkunci",
 "0,99, yaitu cap berukuran tetap. Angka mAP50-95 karena itu sebagian mengukur",
 "kecocokan dengan palet cap, bukan mutu deteksi, dan tidak dapat dinaikkan oleh",
 "model atau augmentasi apa pun.",
 "",
 "Ablasi augmentasi: %s." % verd,
 "",
 "Batas yang melekat.",
 "1. Label adalah kesehatan tajuk generik, BUKAN BSR, tanpa verifikasi lapangan.",
 "2. Tiga ortomosaik satu kebun; tidak ada klaim lintas-kebun atau lintas-kultivar.",
 "3. Kelas Unhealthy bersandar pada ~66 pohon unik: selang kepercayaannya lebar.",
 "4. Angka di papan skor Roboflow TIDAK sebanding: split bawaannya menaruh %.0f%%"
 % (100 * np.mean([r["test_in_train"] for r in la])),
 "   pohon uji di dalam train, jadi ia mengukur hafalan, bukan generalisasi.",
]
print("=" * 78); print("KLAIM MAKSIMUM YANG JUJUR"); print("=" * 78)
print("\n".join(claim))

json.dump(dict(model=MODEL, arm=best_arm, final_tag=FINAL_TAG, setting=st,
               centre=dict(radius_frac=RADIUS_FRAC, conf=CONF,
                           precision=[pr.mean(), sd(pr)], recall=[rc.mean(), sd(rc)],
                           f1=[f1.mean(), sd(f1)], rmse_frac_spacing=[rm.mean(), sd(rm)],
                           per_fold=cen),
               map=[mm.mean(), sd(mm)], map50=[mm50.mean(), sd(mm50)],
               ap50_unhealthy=float(np.nanmean(unh)),
               ablation_conclusive=bool(pos), ablation_delta=abl.get("delta"),
               roboflow_test_in_train=float(np.mean([r["test_in_train"] for r in la])),
               extra_mode=EXTRA_MODE),
          open(os.path.join(y12.BASE, "yolo12_summary.json"), "w"), indent=2, default=float)
print("\ntersimpan: yolo12_summary.json")